# TP3 : Programmation Dynamique

In [ ]:
function readKnaptxtInstance(filename)
    price=Int64[]
    weight=Int64[]
    KnapCap=Int64[]
    open(filename) do f
        for i in 1:3
            tok = split(readline(f))
            if(tok[1] == "ListPrices=")
                for i in 2:(length(tok)-1)
                    push!(price,parse(Int64, tok[i]))
                end
            elseif(tok[1] == "ListWeights=")
                for i in 2:(length(tok)-1)
                    push!(weight,parse(Int64, tok[i]))
                end
            elseif(tok[1] == "Capacity=")
                push!(KnapCap, parse(Int64, tok[2]))
            else
                println("Unknown read :", tok)
            end
        end
    end
    capacity=KnapCap[1]
    return price, weight, capacity
end

## Question 1
La programmation dynamique consiste en la réslution de problèmes de tailles inférieures et l'utilisation des résultats. La reation de récurrence permet de trouver les paramètres des sous-problèmes et de rassembler les résultats. On suppose donc que toute sous-solution d'une solution optimale est une solution optimale du sous-problème.

## Question 2

In [ ]:
filename = "InstancesKnapSack/test.opb.txt"


function resolveKnaptxtInstance(filename, afficher)
    v, w, Q = readKnaptxtInstance(filename)
    n = length(v)

    C = zeros(Q+1,n+1)

    C[1,:] = zeros(n+1) # initialisation de la première colonne (capacité nulle -> valeur nulle)
    # C[j,i]: val max qu'on peut porter si poids capacité est j-1
    # et qu'on a que objets de  1 à i de dispo

    for i = 2:n+1 # disponibilité des objets de 1 à i
        for j = 2:Q+1 # capacité du sac = j - 1
            if w[i-1] <= j-1 # on peut rentrer l'objet i-1
                C1 = C[j-w[i-1], i-1] + v[i-1] # on prend l'objet i-1
                C2 = C[j, i-1]

                if (C1 > C2)
                    # on a pris l'objet i-1
                    C[j,i] = C1
                    # solution du pb de taille j-w[i-1]-1
                    # avec l'objet i-1 en plus
                else
                    # on n'a pas pris l'objet i-1
                    C[j,i] = C2
                end
            else  # pas la place pour l'objet i-1
                C[j,i] = C[j, i-1]
            end
        end
    end

    A = [] # liste des objets pris
    val_max = C[end,end]
    j = Q
    for i in n:-1:1
        if C[j+1,i+1] != C[j+1,i]
            push!(A,i)
            j = j - w[i]
        end
    end
    if (afficher)
        display(C)
        display(A)
    end
    return C[end,end], A
end

C,A = resolveKnaptxtInstance(filename, true)

# Question 3

On atteint une valeur totale de 65 (40+25) pour un poids total de 9 (4+5). La capacité du sac ne nous permettrait pas de mettre plus dde deux objets. L'objet 1 est peu intéressant (très lourd) et l'objet 3 trop peu rentable (interprétation qualitative).

## Question 4

On crée une matrice $C$ de taille $(n+1)\times(Q+1)$. $C_{ji}$ correspond à la valeur maximale qu'on peut porter si la capacité est de $j-1$ et qu'on a droit aux objets de $1$ à $i$. On initialise la première colonne à 0 (capacité nulle donc valeur nulle).

On itère $i$ de $2$ à $n+1$ (on change le nombre d'objets disponibles). Pour chaque liste d'objets disponibles, on itère sur es capacités $j-1$ de $1$ àç $Q$ (les indices en Julia commencent à 1 et on a la colonne d'initialisation).

Ensuite, on teste si on peut rentrer l'objet $i-1$.
- si oui, on compare les valeurs de $C_{j,i-1}$ (on ne prend pas l'objet $i-1$) et $C_{j-w_{i-1},i-1} + v_{i-1}$ (on prend l'objet $i-1$). On assigne à $C_{ji}$ le maximum des deux.
- sinon, la disponibilité de l'objet $i-1$ n'a rien changé, on assigne donc $C_{ji} \leftarrow C_{j,i-1}$.

À la fin de l'algorithme (qui termine car uniquement des "boucles pour"), on lit la valeur maximale dans la dernière case de la dernière ligne de $C$.

## Question 5

On a testé avec tous les sacs fournis, les valeurs obtenues sont bien celles attendues (nom du fichier).

Notre algorithme a l'avantage d'être rapide (complexité temporelle minime), c'est au détriments de la mémoire (complexité spatiale élevée). En effet, la taille de la matrice est $(n+1)\times(Q+1)$, ce qui peut être très grand. Par exemple, pour un très grand sac ($n = 5\ 000$, $Q = 5\ 000\ 000$), on a une erreur de mémoire.

On pourrait alors utiliser une programmation dynamique en mémoire. On ne retient que la colonne de l'itération en cours et la précédente. En effet, l'algorithme peut remonter de plusieurs "lignes" en arrière mais d'une seule colonne ($i$ précédent).

In [ ]:
filenames = readdir("InstancesKnapSack/Weakly_Correlated")

for filename in filenames
    println("fichier : ", filename)
    C,A = resolveKnaptxtInstance("InstancesKnapSack/Weakly_Correlated/"*filename, false)
    println("Valeur max : ", C)
    # println("Objets pris : ", A)
    println("--------------------------------------------")
end

In [ ]:
filenames = readdir("InstancesKnapSack/Uncorrelated")

for filename in filenames
    println("fichier : ", filename)
    C,A = resolveKnaptxtInstance("InstancesKnapSack/Uncorrelated/"*filename, false)
    println("Valeur max : ", C)
    # println("Objets pris : ", A)
    println("--------------------------------------------")
end

In [ ]:
filenames = readdir("InstancesKnapSack/Strongly_Correlated")

for filename in filenames
    println("fichier : ", filename)
    C,A = resolveKnaptxtInstance("InstancesKnapSack/Strongly_Correlated/"*filename, false)
    println("Valeur max : ", C)
    # println("Objets pris : ", A)
    println("--------------------------------------------")
end

In [ ]:
filenames = readdir("InstancesKnapSack/Similar_Weights")

for filename in filenames
    println("fichier : ", filename)
    C,A = resolveKnaptxtInstance("InstancesKnapSack/Similar_Weights/"*filename, false)
    println("Valeur max : ", C)
    # println("Objets pris : ", A)
    println("--------------------------------------------")
end

## Question 6

Le TP2 n'a pas été réalisé, mais on compte environ $18s$ pour tous les sacs fournis (lecture des dossiers et parsing des fichiers inclus). Ce qui est très rapide étant donné la taille des exemples.